# 04 — Train: persistence baseline

Evaluates a 24-hour persistence baseline for the configured target station on the test feature artifact.

**Inputs:** train-derived and test-derived feature artifacts  
**Outputs:** in-notebook prediction preview and test metrics only

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and pins the notebook's constants. There is no model configuration to pin: persistence has no hyperparameters and nothing to fit.

**What the imports provide**

- `mean_absolute_error`, `root_mean_squared_error` — the two reported error metrics, in the water level's native unit.
- `feature_column_names()`, `target_column_names()` — the Stage-3 column contract, used here so that this baseline is scored on exactly the same cohort as the fitted candidates.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed` | Directory the Stage-3 feature Parquets are read from. This notebook only reads — it never writes back. |
| `PREDICTION_PREVIEW_ROWS` | `5` | How many scored test rows the final preview table shows. Display only; it has no effect on any metric. |
| `FEATURE_COLUMNS` | 53 names | The frozen Stage-3 predictor contract, read from `feature_column_names()` instead of being hardcoded so the notebook fails loudly if Stage 3 ever changes it. It contains: the raw `water_level`, `imputed`, `precipitation` and `temperature_2m`; 8 water-level lags (1, 3, 6, 12, 24, 48, 72, 168 h); 5 water-level differences (1–24 h); 16 rolling water-level statistics (mean/std/min/max x 6/24/72/168 h); 4 rolling imputation counts; 4 rolling precipitation sums; 4 rolling temperature means plus the 24 h temperature min and max; and 6 calendar Fourier terms. |
| `TARGET_COLUMNS` | `target_t_plus_01` … `target_t_plus_24` | The 24 future water levels, one per lead hour. The model emits all of them from a single feature vector — a *direct* multi-horizon setup, with no recursive feeding of its own predictions. |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from src.config import TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

PROCESSED_DIR = Path("data/processed")
PREDICTION_PREVIEW_ROWS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())

## Shared evaluation cohort

Every stage-4 candidate is fit and scored on exactly the same rows, which is what makes their reported numbers comparable to each other and to the persistence baseline. One row is one *issue time* `t`, and it qualifies only when both of these hold:

1. **Stage 3 marked it `target_valid`.** All 24 future water levels `t+1 … t+24` were actually observed, none of them synthesised. This drops issue times sitting near a data gap, plus the final 24 hours of each artifact, which have no complete future.
2. **All 53 predictors are present.** The lag and rolling features need a complete 168-hour lookback, so the first week of each artifact is a warm-up that can never qualify.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it.

Persistence has no fitted parameters, so the train artifact is loaded and cohort-checked purely to confirm that both splits satisfy the Stage-3 contract and to keep this notebook's eligibility logic byte-for-byte identical to the model notebooks. Not a single train value reaches the forecast.

## Helper functions

Three small helpers, identical to the ones in the model notebooks so that the numbers line up exactly. All of them take keyword-only `station_id` and `artifact_name` arguments that exist purely so a raised error names the split it came from instead of leaving you to guess.

**`eligible_rows(frame, *, station_id, artifact_name) -> pd.Series`**

- `frame` — one loaded feature artifact (train or test).
- `station_id` — the station the artifact is supposed to describe; used in error messages.
- `artifact_name` — `"train"` or `"test"`, likewise for error messages.

Returns a boolean mask marking the cohort rows defined above. It raises instead of returning a mask when the artifact is missing a contract column, or when a `target_valid` row still carries a null target — that combination is a Stage-3 bug, and silently averaging around it would produce a metric that looks fine and is not.

**`metric_tables(actual, predictions, *, station_id) -> (aggregate, per_horizon)`**

- `actual` — the cohort's `TARGET_COLUMNS` frame, shape `(n_issue_times, 24)`.
- `predictions` — the model's output, in the same shape and the same row order.

Returns two frames. The **aggregate** one pools all `n x 24` values into a single MAE and RMSE. The **per-horizon** one repeats that calculation separately for each lead time, which is what reveals how fast accuracy decays from `t+1` to `t+24`. Both metrics are in the water level's native unit: MAE is the average absolute miss, while RMSE squares the errors before averaging, so it is always at least as large as MAE and is dominated by the worst forecasts — a wide gap between the two means a few large misses rather than uniformly poor accuracy.

**`prediction_preview(frame, predictions) -> pd.DataFrame`**

- `frame` — the scored cohort rows, supplying `timestamp` and the actual targets.
- `predictions` — the matching predicted array.

Returns one frame with each `prediction_target_t_plus_XX` column beside its actual counterpart, aligned on the cohort's original index so no row silently shifts.

In [ ]:
def eligible_rows(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> pd.Series:
    """Return model-ready rows and reject incomplete feature artifacts."""
    required_columns = {"timestamp", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    eligible = frame["target_valid"].eq(True) & frame[FEATURE_COLUMNS].notna().all(
        axis=1
    )
    if frame.loc[eligible, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return eligible


def metric_tables(
    actual: pd.DataFrame, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
                "rmse": root_mean_squared_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[target], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(
                    actual[target], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon


def prediction_preview(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    """Return issue timestamps, actual targets, and direct multi-step predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load, forecast, and evaluate

One cell, because a baseline with nothing to fit does not need more.

1. **Load.** Resolve `data/processed/<station>_train_features.parquet` and `<station>_test_features.parquet` for `src.config.TARGET_STATION_ID`, raising `FileNotFoundError` if either is absent.
2. **Filter.** Build both eligibility masks and stop early if either split has no usable row. Only `test_rows` is carried forward.
3. **Forecast.** The whole model is this: *whatever the water level is now, that is the forecast for all 24 hours ahead.* `np.repeat(test_rows[["water_level"]].to_numpy(), repeats=len(TARGET_COLUMNS), axis=1)` takes the single observed water-level column at each issue time and tiles it across 24 columns, producing the same `(n_issue_times, 24)` shape every other candidate produces.
   - `repeats=len(TARGET_COLUMNS)` — 24, one copy per lead hour.
   - `axis=1` — repeat along columns (horizons), not rows. `axis=0` would instead duplicate issue times and silently break the alignment with `test_rows`.
4. **Score.** The same `metric_tables` and `prediction_preview` as everywhere else.

This is the reference every other stage-4 notebook has to beat. River level is smooth and strongly autocorrelated, which makes persistence genuinely hard to improve on at `t+1` and progressively easier towards `t+24` — so the per-horizon table matters more than the aggregate one here. A candidate that cannot beat these numbers has not learned anything worth keeping.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
for artifact_path in (train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing feature artifact for {station_id}: {artifact_path}"
        )

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)
train_mask = eligible_rows(train_features, station_id=station_id, artifact_name="train")
test_mask = eligible_rows(test_features, station_id=station_id, artifact_name="test")
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

test_rows = test_features.loc[test_mask]
test_predictions = np.repeat(
    test_rows[["water_level"]].to_numpy(),
    repeats=len(TARGET_COLUMNS),
    axis=1,
)

aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS], test_predictions, station_id=station_id
)
print(f"Persistence test results for {station_id}")
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))